# Wiener Linien Fleet Spotter — Dataset EDA & Annotation Check

This notebook helps you:
1. Visualise the class distribution in the annotated dataset
2. Spot-check bounding box annotations
3. Preview augmented samples
4. Evaluate a trained model interactively

In [ ]:
import os, random
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import yaml

ROOT       = Path("..")
DATA_YAML  = ROOT / "data" / "dataset.yaml"

with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)

CLASS_NAMES = list(cfg["names"].values())
COLORS = ['#E63946','#457B9D','#2A9D8F','#6C757D','#C1121F','#7209B7']
print("Classes:", CLASS_NAMES)

## 1. Class Distribution

In [ ]:
def count_classes(label_dir: Path):
    counts = {n: 0 for n in CLASS_NAMES}
    for lbl in label_dir.glob("*.txt"):
        for line in lbl.read_text().strip().splitlines():
            cls = int(line.split()[0])
            if cls < len(CLASS_NAMES):
                counts[CLASS_NAMES[cls]] += 1
    return counts

splits = ['train', 'val', 'test']
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, split in zip(axes, splits):
    lbl_dir = ROOT / 'data' / 'annotated' / split / 'labels'
    if not lbl_dir.exists():
        ax.set_title(f'{split} (no data yet)')
        ax.text(0.5, 0.5, 'No annotations found', ha='center', transform=ax.transAxes)
        continue
    counts = count_classes(lbl_dir)
    ax.bar(counts.keys(), counts.values(), color=COLORS)
    ax.set_title(f'{split.capitalize()} split')
    ax.set_xticklabels(counts.keys(), rotation=30, ha='right')
    ax.set_ylabel('# instances')

plt.suptitle('Class Distribution per Split', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(ROOT / 'assets' / 'class_distribution.png', dpi=150)
plt.show()

## 2. Annotation Visualisation

In [ ]:
def draw_yolo_boxes(image_path, label_path, class_names, colors):
    img = cv2.imread(str(image_path))
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 6))
    ax.imshow(img)
    
    if label_path.exists():
        for line in label_path.read_text().strip().splitlines():
            parts = line.split()
            if len(parts) != 5:
                continue
            cls, cx, cy, bw, bh = int(parts[0]), *[float(x) for x in parts[1:]]
            x1 = (cx - bw/2) * w
            y1 = (cy - bh/2) * h
            rect = patches.Rectangle((x1, y1), bw*w, bh*h,
                                       linewidth=2, edgecolor=colors[cls % len(colors)],
                                       facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, y1-5, class_names[cls], color='white', fontsize=9,
                    bbox=dict(boxstyle='round,pad=0.2', facecolor=colors[cls % len(colors)]))
    
    ax.axis('off')
    ax.set_title(image_path.name)
    return fig

# Show 4 random training images
img_dir = ROOT / 'data' / 'annotated' / 'train' / 'images'
lbl_dir = ROOT / 'data' / 'annotated' / 'train' / 'labels'

if img_dir.exists():
    images = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))
    sample = random.sample(images, min(4, len(images)))
    for img_path in sample:
        draw_yolo_boxes(img_path, lbl_dir / (img_path.stem + '.txt'), CLASS_NAMES, COLORS)
        plt.show()
else:
    print("No training images found yet — add images to data/annotated/train/images/")

## 3. Model Inference Demo (after training)

In [ ]:
from ultralytics import YOLO

weights = ROOT / 'model' / 'runs' / 'intensive-100ep-541img' / 'weights' / 'best.pt'

if weights.exists():
    model = YOLO(str(weights))
    
    # Run on test images
    test_dir = ROOT / 'data' / 'annotated' / 'test' / 'images'
    if test_dir.exists():
        test_images = list(test_dir.glob('*.jpg'))[:4]
        for img_path in test_images:
            result = model.predict(str(img_path), conf=0.3, verbose=False)[0]
            ann = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)
            plt.figure(figsize=(10,6))
            plt.imshow(ann)
            plt.title(img_path.name)
            plt.axis('off')
            plt.show()
else:
    print("Model not trained yet. Run: python scripts/train.py")